# Detectarea textelor human / AI în limba română

Selectează **Runtime > Change runtime type > T4 GPU**, într-o sesiune nouă, apoi **Run all**.
Datele sunt descărcate automat din GitHub; nu trebuie generat nimic local și nu este necesar Drive.
Se antrenează un model nou, se calibrează și se evaluează pe ambele loturi existente.
Rularea poate dura câteva ore. La final se descarcă o arhivă cu modelul și rezultatele.

Notebookul este configurat pentru mediul T4 al experimentului. Dacă GPU-ul sau dependențele
nu sunt compatibile, execuția se oprește explicit. Scorurile unei rulări noi pot diferi de cele
din raport. Această versiune a fost verificată local pentru structură și date, nu încă prin
antrenare integrală într-o sesiune GPU nouă.


## 1. Configurație și verificarea GPU-ului

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import uuid, sys, subprocess, json, hashlib, zipfile, types

gpu_check = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True) if __import__('shutil').which('nvidia-smi') else None
if gpu_check is None or gpu_check.returncode or 'T4' not in gpu_check.stdout:
    raise RuntimeError('Selectează T4 GPU în Runtime > Change runtime type, apoi folosește o sesiune nouă.')
if any(name in sys.modules for name in ('torch', 'transformers', 'unsloth')):
    raise RuntimeError('Folosește o sesiune Python nouă înainte de Run all; bibliotecile sunt deja importate.')

OFFLINE_VALIDATION = False
RUN_MODE = 'train'
RUN_NAME = 'qwen35_4b_ro_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8]
MODEL_NAME = 'unsloth/Qwen3.5-4B'
MAX_SEQ_LENGTH = 12288
SEED = 3407
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 16
NUM_TRAIN_EPOCHS = 1
LEARNING_RATE = 2e-4
ENABLE_T4_KV_FIX = True
CONFIRM_FULL_TRAINING = True
RESUME_CHECKPOINT = None
WORK_ROOT = Path('/content/SDCGIA')
SAVE_ROOT = WORK_ROOT / 'runs'
BUNDLE_PATH = WORK_ROOT / 'date/SDCGIA_human_ai_v1.zip'
SUPPLEMENT_BUNDLE = WORK_ROOT / 'date/supplementary_evaluation.zip'
EXTRACT_ROOT = WORK_ROOT / 'dataset'
EXPECTED_MANIFEST_SHA = '28ce73f01101e43c8285144dacb77921f387412efcd2dd10ad2731b78dc1a8ae'
print('GPU:', gpu_check.stdout.strip(), '| Rulare nouă:', RUN_NAME)


## 2. Instalare

Se fixează versiunile principale folosite în experiment și sursa Unsloth verificată.
Nu rula o altă celulă de instalare peste acest mediu. Într-o sesiune nouă, bibliotecile
sunt importate doar după instalare; dacă Colab solicită restart, repornește sesiunea și reia de la început.


In [ ]:
UNSLOTH_REVISION = '3594ed54db37538c2a7fe975a3d12ac495c0cb6a'
EXPECTED_DECISION_SHA = '1b4a9d45965e4cfa93995d634d2104637f2c559595f06b104cc1e94075a45b67'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
constraints = WORK_ROOT / 'constraints.txt'
constraints.write_text('\n'.join([
    'torch==2.8.0', 'torchvision==0.23.0', 'torchaudio==2.8.0',
    'transformers==5.2.0', 'unsloth_zoo==2026.10.7', 'peft==0.21.2',
    'trl==0.22.2', 'bitsandbytes==0.50.2', 'tokenizers>=0.22.0,<=0.23.0',
    'xformers==0.0.32.post2', 'torchao==0.16.0'
]) + '\n', encoding='utf-8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
    'torch==2.8.0', 'torchvision==0.23.0', 'torchaudio==2.8.0',
    '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-c', str(constraints),
    'unsloth[base] @ git+https://github.com/unslothai/unsloth.git@' + UNSLOTH_REVISION,
    'unsloth_zoo[base]==2026.10.7', 'transformers==5.2.0', 'peft==0.21.2',
    'trl==0.22.2', 'bitsandbytes==0.50.2', 'xformers==0.0.32.post2'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', 'torchao==0.16.0'], check=True)
import os
kernel_env = os.environ.copy()
kernel_env['PIP_CONSTRAINT'] = str(constraints)
kernel_env['UV_CONSTRAINT'] = str(constraints)
subprocess.run(['unsloth', 'install-kernels'], check=True, env=kernel_env)
import importlib.metadata as package_metadata
decision_file = Path(package_metadata.distribution('unsloth').locate_file('unsloth/models/decision.py'))
assert hashlib.sha256(decision_file.read_bytes()).hexdigest() == EXPECTED_DECISION_SHA, 'Sursa Unsloth diferă'
expected_versions = {'torch':'2.8.0', 'transformers':'5.2.0', 'unsloth':'2026.10.7',
                     'unsloth_zoo':'2026.10.7', 'peft':'0.21.2', 'trl':'0.22.2', 'bitsandbytes':'0.50.2'}
for name, expected in expected_versions.items():
    actual = package_metadata.version(name).split('+')[0]
    assert actual == expected, f'{name}: {actual}, așteptat {expected}'
print('Instalare terminată; versiunile principale și sursa Unsloth sunt verificate.')


## 3. Descărcarea automată a datelor

In [ ]:
import urllib.request

DATA_BASE = 'https://raw.githubusercontent.com/Neculai13/SDCGIA/main/date/'
def download_verified(url, target, expected_sha):
    target = Path(target)
    if target.exists():
        assert hashlib.sha256(target.read_bytes()).hexdigest() == expected_sha, 'Fișier local diferit: ' + target.name
        return target
    with urllib.request.urlopen(url, timeout=120) as response:
        raw = response.read()
    assert hashlib.sha256(raw).hexdigest() == expected_sha, 'Descărcare invalidă: ' + target.name
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_suffix('.download')
    temporary.write_bytes(raw)
    temporary.replace(target)
    return target

download_verified(DATA_BASE + BUNDLE_PATH.name, BUNDLE_PATH,
    '098ae67d54b2c94cbf9d4e585c83a16c25a249abc0d0bc8c3ab253a000876562')
download_verified(DATA_BASE + SUPPLEMENT_BUNDLE.name, SUPPLEMENT_BUNDLE,
    'bf20cf6aabc15a262a7d54d2efb77bea7c01a8dbfb425aa9afe0cbfbca752c20')
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    assert archive.testzip() is None
    manifest_bytes = archive.read('bundle_manifest.json')
    assert hashlib.sha256(manifest_bytes).hexdigest() == EXPECTED_MANIFEST_SHA
    bundle_manifest = json.loads(manifest_bytes)
    runtime_bytes = archive.read('colab_human_ai_runtime.py')
    assert hashlib.sha256(runtime_bytes).hexdigest() == bundle_manifest['files']['colab_human_ai_runtime.py']
runtime = types.ModuleType('sdcgia_runtime')
exec(compile(runtime_bytes, 'colab_human_ai_runtime.py', 'exec'), runtime.__dict__)
runtime.unpack_verified(BUNDLE_PATH, EXTRACT_ROOT, EXPECTED_MANIFEST_SHA)
data, metadata = runtime.load_splits(EXTRACT_ROOT)
assert {k:len(v) for k,v in data.items()} == {'train':4448, 'calibration':955, 'test':954}

with zipfile.ZipFile(SUPPLEMENT_BUNDLE) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == {'manifest.json', 'protocol.json', 'evaluation.jsonl'}
    supplement_manifest = json.loads(archive.read('manifest.json'))
    for name, digest in supplement_manifest['files'].items():
        assert hashlib.sha256(archive.read(name)).hexdigest() == digest
    supplement_protocol = json.loads(archive.read('protocol.json'))
    supplement_rows = [json.loads(s) for s in archive.read('evaluation.jsonl').decode().splitlines()]
assert len(supplement_rows) == len({r['id'] for r in supplement_rows}) == 150
assert sum(r['label']=='human' for r in supplement_rows) == 30
assert sum(r['label']=='ai' for r in supplement_rows) == 120
original_sources = {r['source_id'] for values in metadata.values() for r in values}
assert not original_sources & {r['source_id'] for r in supplement_rows}
assert all(r['label'] in {'human','ai'} and r['text'].strip() and
    hashlib.sha256(r['text'].encode()).hexdigest() == r['text_sha256'] for r in supplement_rows)
print('Date principale:', {k:len(v) for k,v in data.items()}, '| Lot suplimentar:', len(supplement_rows))


## 4. Tokenizare integrală, fără trunchiere

In [ ]:
if not OFFLINE_VALIDATION:
    import unsloth  # înainte de transformers/torch
    from transformers import AutoTokenizer
    from unsloth.models.clef import encode_record
    tokenizer_audit = AutoTokenizer.from_pretrained(MODEL_NAME)
    def audit_encode(row, budget):
        return encode_record(tokenizer_audit, row, max_length=budget)
    length_report = runtime.inspect_lengths(data, metadata, audit_encode, MAX_SEQ_LENGTH)
    analysis_dir = SAVE_ROOT / RUN_NAME / ("analysis_ctx" + str(MAX_SEQ_LENGTH))
    runtime.save_json(analysis_dir / "token_lengths.json", length_report)
    print(json.dumps({k: v for k, v in length_report.items() if k != "records"}, indent=2))
    CONTEXT_OK = length_report["over_context"] == 0
    print("Context suficient:", CONTEXT_OK)
    if not CONTEXT_OK:
        print("OPRIRE DE SIGURANȚĂ: pilotul/antrenarea sunt blocate, nu se acceptă trunchiere implicită.")
else:
    CONTEXT_OK = False
    print("OFFLINE: tokenizarea reală nu este verificată; nu există rezultat GPU sau de lungime în tokeni.")


## 5. Atenție eficientă pe T4

Este aceeași corecție restrânsă folosită în experiment: replicarea explicită a capetelor K/V
prin calea existentă Transformers. Versiunile și GPU-ul sunt verificate înainte de aplicare.


In [ ]:
"""Narrow, reversible Transformers 5.2 / PyTorch 2.8 T4 GQA workaround.

No imports of torch/transformers at import, no disk changes, no weight changes.
The original HF sdpa_attention_forward performs repeat_kv when this selector
returns False. Its mask/dropout/scale/causal handling stays unchanged.
"""
import hashlib
import inspect
from pathlib import Path

POLICY = 't4_explicit_kv_transformers_selector_v1'
EXPECTED_VERSIONS = {'torch': '2.8.0', 'transformers': '5.2.0',
                     'unsloth': '2026.10.7', 'unsloth_zoo': '2026.10.7'}


class T4ExplicitKVFix:
    def __init__(self, torch_module, sdpa_module, versions):
        for name, expected in EXPECTED_VERSIONS.items():
            actual = versions.get(name, '').split('+')[0]
            if actual != expected:
                raise RuntimeError(f'Runtime nevalidat pentru corecție: {name}={actual}; așteptat {expected}.')
        if tuple(torch_module.cuda.get_device_capability(0)) != (7, 5):
            raise RuntimeError('Corecția este restrânsă la GPU Turing (compute capability 7.5).')
        original = getattr(sdpa_module, 'use_gqa_in_sdpa', None)
        if not callable(original) or not callable(getattr(sdpa_module, 'repeat_kv', None)):
            raise RuntimeError('API Transformers diferit; corecția nu a fost aplicată.')
        if hasattr(original, '_sdcgia_fix'):
            raise RuntimeError('Corecția este deja aplicată. Repornește sesiunea înainte de rerulare.')
        if list(inspect.signature(original).parameters) != ['attention_mask', 'key']:
            raise RuntimeError('Semnătura selectorului GQA diferă de cea verificată.')
        self.torch, self.module, self.original = torch_module, sdpa_module, original
        self.calls, self.shapes = 0, set()
        self.versions = {k: versions[k] for k in EXPECTED_VERSIONS}
        try:
            source = Path(inspect.getsourcefile(original)).read_bytes()
            self.source_sha256 = hashlib.sha256(source).hexdigest()
        except (OSError, TypeError):
            self.source_sha256 = None

        def selector(attention_mask, key):
            targeted = (attention_mask is None and key.device.type == 'cuda'
                        and key.ndim == 4 and key.shape[1] == 4 and key.shape[-1] == 256
                        and key.dtype == self.torch.float16
                        and tuple(self.torch.cuda.get_device_capability(key.device)) == (7, 5))
            if targeted:
                self.calls += 1
                if len(self.shapes) < 32:
                    self.shapes.add(tuple(int(n) for n in key.shape))
                return False
            return self.original(attention_mask, key)

        selector._sdcgia_fix = self
        self.selector = selector
        self.module.use_gqa_in_sdpa = selector

    def assert_installed(self):
        if self.module.use_gqa_in_sdpa is not self.selector:
            raise RuntimeError('Selectorul de atenție a fost înlocuit ulterior; nu continua pilotul.')

    def reset_counters(self):
        self.calls, self.shapes = 0, set()

    def config(self):
        return {'policy': POLICY, 'scope': 'T4 CUDA float16 K/V heads=4 head_dim=256 mask=None',
                'versions': self.versions, 'transformers_sdpa_source_sha256': self.source_sha256,
                'model_weights_changed': False, 'dtype_changed': False, 'article_text_changed': False,
                'selector': 'explicit_repeat_kv_via_existing_transformers_path'}

    def snapshot(self):
        return {**self.config(), 'bypassed_native_gqa_calls': self.calls,
                'observed_key_shapes_before_repeat': [list(s) for s in sorted(self.shapes)]}

    def restore(self):
        self.assert_installed()
        self.module.use_gqa_in_sdpa = self.original


def inspect_attention_events(events):
    """Inspect executed ATen operators, not merely available/enabled backend flags."""
    operators = {event.key: int(event.count) for event in events
                 if 'scaled_dot_product' in event.key.lower()}
    forward = operators.get('aten::_scaled_dot_product_efficient_attention', 0)
    backward = operators.get('aten::_scaled_dot_product_efficient_attention_backward', 0)
    math_calls = sum(n for key, n in operators.items()
                     if key.startswith('aten::') and 'scaled_dot_product_attention_math' in key)
    return {'operators': operators, 'efficient_forward_calls': forward,
            'efficient_backward_calls': backward, 'math_sdpa_calls': math_calls,
            'efficient_forward_and_backward_observed': forward > 0 and backward > 0,
            'no_math_sdpa_observed': math_calls == 0,
            'scope': 'CPU profiler observes dispatched ATen operators during the two real pilot steps; no tensor shapes/stacks retained.'}


def require_efficient_pilot(report, bypassed_calls):
    if bypassed_calls <= 0:
        raise RuntimeError('Corecția nu a fost traversată. Pilotul nu este confirmat.')
    if not report['efficient_forward_and_backward_observed'] or not report['no_math_sdpa_observed']:
        raise RuntimeError('Profilerul nu confirmă ruta memory-efficient fără SDPA math. Trimite raportul; nu porni antrenarea completă.')

ATTENTION_FIX_SOURCE_SHA = '237106bbac04388e8a126a5e78331b5edc7f0e1b6d8756a4c9992633911aad12'
attention_fix = None
if not OFFLINE_VALIDATION and RUN_MODE != "prepare" and ENABLE_T4_KV_FIX:
    if MODEL_NAME != "unsloth/Qwen3.5-4B":
        raise RuntimeError("Această corecție este verificată doar pentru Qwen3.5-4B/T4; nu schimba modelul aici.")
    import torch, importlib, importlib.metadata as package_metadata
    sdpa_module = importlib.import_module("transformers.integrations.sdpa_attention")
    patch_versions = {name: package_metadata.version(name) for name in EXPECTED_VERSIONS}
    attention_fix = T4ExplicitKVFix(torch, sdpa_module, patch_versions)
    print("Corecție activă:", attention_fix.config())
else:
    print("Corecția nu este activată în prepare/offline sau este dezactivată explicit.")


## 6. Încărcarea modelului de bază și LoRA nou

In [ ]:
if not OFFLINE_VALIDATION and RUN_MODE != "prepare":
    if not CONTEXT_OK:
        raise RuntimeError("Context insuficient. Trimite raportul; nu rula antrenarea cu texte tăiate.")
    if RUN_MODE == "train" and not CONFIRM_FULL_TRAINING:
        raise RuntimeError("După pilot, setează CONFIRM_FULL_TRAINING=True explicit.")
    if "model" in globals():
        raise RuntimeError("Repornește sesiunea înainte de încărcarea repetată a modelului.")
    from unsloth import FastDecisionModel, DecisionTrainer, is_bfloat16_supported
    from transformers import set_seed
    import torch
    assert torch.cuda.is_available()
    set_seed(SEED)
    phase = "pilot_" + RUN_NAME if RUN_MODE == "pilot" else RUN_NAME
    RUN_DIR = SAVE_ROOT / phase
    if RUN_MODE == "train" and (RUN_DIR / "training_complete.json").exists():
        raise RuntimeError("Antrenarea este deja încheiată. Folosește calibrate/evaluate sau un RUN_NAME nou.")
    if RUN_MODE == "pilot" and (RUN_DIR / "pilot_complete.json").exists():
        raise RuntimeError("Pilotul este deja încheiat. Folosește un RUN_NAME nou pentru alt pilot.")
    if RUN_MODE == "train" and RESUME_CHECKPOINT is None and list((RUN_DIR / "checkpoints").glob("checkpoint-*")):
        raise RuntimeError("Există checkpointuri. Specifică RESUME_CHECKPOINT sau folosește un RUN_NAME nou.")
    load_path = str(RUN_DIR / "model_calibrated") if RUN_MODE == "evaluate" else str(RUN_DIR / "model_uncalibrated") if RUN_MODE == "calibrate" else MODEL_NAME
    if RUN_MODE == "evaluate" and not (RUN_DIR / "calibration_complete.json").exists():
        raise RuntimeError("Lipsește modelul calibrat al acestui experiment.")
    if RUN_MODE == "calibrate" and not (RUN_DIR / "training_complete.json").exists():
        raise RuntimeError("Antrenarea completă nu este salvată; reia checkpointul în modul train.")
    model, tokenizer = FastDecisionModel.from_pretrained(model_name=load_path,
        max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
    if RUN_MODE in {"pilot", "train"}:
        model = FastDecisionModel.get_peft_model(model, r=16, lora_alpha=16,
            lora_dropout=0, use_gradient_checkpointing="unsloth", random_state=SEED)
    assert getattr(model, "is_clef", False), "API schimbat: acest notebook este verificat pentru Clef/Qwen3.5"
    items = runtime.strict_items(data, metadata, tokenizer, model, FastDecisionModel, length_report)
    run_config = {"model": MODEL_NAME, "max_seq_length": MAX_SEQ_LENGTH,
        "seed": SEED, "batch_size": TRAIN_BATCH_SIZE, "gradient_accumulation": 1 if RUN_MODE == "pilot" else GRADIENT_ACCUMULATION,
        "epochs": NUM_TRAIN_EPOCHS, "learning_rate": LEARNING_RATE,
        "lora_rank": 16, "bundle_manifest_sha256": EXPECTED_MANIFEST_SHA,
        "phase": "pilot" if RUN_MODE == "pilot" else "full", "no_truncation": True}
    import importlib.metadata as package_metadata, sys, inspect
    from unsloth.models import decision as decision_module
    versions = {}
    for package in ("unsloth", "unsloth_zoo", "torch", "transformers", "peft", "trl", "bitsandbytes"):
        try: versions[package] = package_metadata.version(package)
        except package_metadata.PackageNotFoundError: versions[package] = "unknown"
    run_config["runtime_versions"] = versions
    run_config["decision_source_sha256"] = runtime.sha_bytes(Path(inspect.getfile(decision_module)).read_bytes())
    run_config["base_revision"] = model.decision_config.get("base_revision")
    run_config["attention_fix"] = attention_fix.config() if attention_fix else {"policy": "original"}
    run_config["attention_fix_source_sha256"] = ATTENTION_FIX_SOURCE_SHA
    runtime.freeze_run(RUN_DIR, run_config)
    runtime.save_json(RUN_DIR / ("environment_" + RUN_MODE + ".json"), {
        "versions": versions, "python": sys.version, "gpu": torch.cuda.get_device_name(0),
        "decision_source_sha256": run_config["decision_source_sha256"],
        "decision_config": model.decision_config})
    print("Conversie integrală OK:", {k: len(v) for k, v in items.items()})
else:
    print("Încărcarea modelului și antrenarea sunt inactive în modul prepare/offline.")


## 7. Antrenare de la zero

Se folosesc numai cele 4.448 de exemple train. Checkpointurile sunt locale și nu se reia
checkpointul altei rulări. O sesiune Colab întreruptă poate pierde fișierele din /content.


In [ ]:
if not OFFLINE_VALIDATION and RUN_MODE in {"pilot", "train"}:
    from transformers import TrainingArguments
    import time
    if RUN_MODE == "pilot":
        selected = sorted(range(len(items["train"])), key=lambda i: len(items["train"][i]["input_ids"]))
        indexes = selected[-2:]
        training_items = [items["train"][i] for i in indexes]
    else:
        training_items = items["train"]
    if RESUME_CHECKPOINT is not None:
        checkpoint = Path(RESUME_CHECKPOINT).resolve()
        if checkpoint.parent != (RUN_DIR / "checkpoints").resolve() or not (checkpoint / "trainer_state.json").exists():
            raise RuntimeError("Checkpointul nu aparține acestui experiment sau nu conține starea trainerului.")
    arguments = TrainingArguments(output_dir=str(RUN_DIR / "checkpoints"),
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=1 if RUN_MODE == "pilot" else GRADIENT_ACCUMULATION,
        num_train_epochs=NUM_TRAIN_EPOCHS, max_steps=2 if RUN_MODE == "pilot" else -1,
        warmup_steps=0 if RUN_MODE == "pilot" else 10, learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine", weight_decay=.01,
        bf16=is_bfloat16_supported(), fp16=not is_bfloat16_supported(),
        eval_strategy="no", logging_steps=1, save_strategy="steps", save_steps=10,
        save_total_limit=2, save_only_model=False, report_to="none", seed=SEED)
    trainer = DecisionTrainer(model=model, processing_class=tokenizer,
        train_dataset=training_items, args=arguments)
    if attention_fix:
        attention_fix.assert_installed()
        attention_fix.reset_counters()
    if RUN_MODE == "pilot":
        assert TRAIN_BATCH_SIZE == 1 and len(training_items) == 2
        pilot_selection = [{"id": metadata["train"][i]["id"],
                            "tokens": len(items["train"][i]["input_ids"])} for i in indexes]
        runtime.save_json(RUN_DIR / "pilot_selection.json", pilot_selection)
        print("Articole pilot:", pilot_selection)
    torch.cuda.reset_peak_memory_stats()
    started = time.monotonic()
    from contextlib import nullcontext
    from torch.profiler import profile, ProfilerActivity
    profiling = profile(activities=[ProfilerActivity.CPU], record_shapes=False,
                        profile_memory=False, with_stack=False) if RUN_MODE == "pilot" else nullcontext()
    try:
        with profiling as attention_profile:
            training_stats = trainer.train(resume_from_checkpoint=RESUME_CHECKPOINT)
        torch.cuda.synchronize()
        if RUN_MODE == "pilot":
            backend_report = inspect_attention_events(attention_profile.key_averages())
            backend_report["attention_fix"] = attention_fix.snapshot() if attention_fix else None
            runtime.save_json(RUN_DIR / "attention_backend_report.json", backend_report)
            print(json.dumps(backend_report, indent=2))
            if attention_fix:
                require_efficient_pilot(backend_report, attention_fix.calls)
    except Exception as pilot_error:
        runtime.save_json(RUN_DIR / ("pilot_failed.json" if RUN_MODE == "pilot" else "training_failed.json"), {
            "exception_type": type(pilot_error).__name__, "message": str(pilot_error),
            "attention_fix": attention_fix.snapshot() if attention_fix else None,
            "peak_reserved_gib": torch.cuda.max_memory_reserved()/1024**3})
        raise
    training_seconds = time.monotonic()-started
    if not all(torch.isfinite(parameter).all().item() for parameter in model.parameters() if parameter.requires_grad):
        raise RuntimeError("Parametri antrenabili ne-finiți după pilot/antrenare; nu se marchează complet.")
    model.save_pretrained(str(RUN_DIR / "model_uncalibrated"))
    completed = {"attention_fix": attention_fix.snapshot() if attention_fix else None,
                 "mode": RUN_MODE, "training_metrics": training_stats.metrics,
                 "seconds": training_seconds,
                 "peak_reserved_gib": torch.cuda.max_memory_reserved()/1024**3}
    runtime.save_json(RUN_DIR / ("pilot_complete.json" if RUN_MODE == "pilot" else "training_complete.json"), completed)
    print(json.dumps(completed, indent=2))
    if RUN_MODE == "pilot":
        print("Pilot terminat. NU continua spre evaluarea finală. Repornește sesiunea înainte de train.")
else:
    print("Antrenarea nu este activată.")


## 8. Calibrare pe cele 955 de exemple dedicate

In [ ]:
if not OFFLINE_VALIDATION and RUN_MODE in {"train", "calibrate"}:
    if not (RUN_DIR / "training_complete.json").exists():
        raise RuntimeError("Antrenarea completă nu este confirmată.")
    calibration_metrics = FastDecisionModel.calibrate(model, tokenizer, items["calibration"], batch_size=1)
    runtime.save_json(RUN_DIR / "calibration_metrics.json", calibration_metrics)
    model.save_pretrained(str(RUN_DIR / "model_calibrated"))
    saved_files = list((RUN_DIR / "model_calibrated").iterdir())
    assert any(p.name.startswith("adapter_model") for p in saved_files)
    assert (RUN_DIR / "model_calibrated" / "joint_head.safetensors").exists(), "Decision head lipsă"
    final_fingerprint = runtime.adapter_fingerprint(RUN_DIR / "model_calibrated")
    runtime.save_json(RUN_DIR / "calibration_complete.json", {"adapter_fingerprint": final_fingerprint})
    print("Model calibrat salvat în:", RUN_DIR / "model_calibrated")
elif not OFFLINE_VALIDATION and RUN_MODE == "evaluate":
    final_fingerprint = runtime.adapter_fingerprint(RUN_DIR / "model_calibrated")
    saved = json.loads((RUN_DIR / "calibration_complete.json").read_text())
    assert saved["adapter_fingerprint"] == final_fingerprint
    print("Model calibrat reîncărcat; configurația și adapterul sunt verificate.")
else:
    print("Calibrarea nu este activată.")


## 9. Evaluare pe testul principal

In [ ]:
RUN_FINAL_TEST = True  # Schimbă în True numai după antrenare și calibrare.
if not OFFLINE_VALIDATION and RUN_FINAL_TEST:
    if RUN_MODE not in {"train", "calibrate", "evaluate"} or not (RUN_DIR / "calibration_complete.json").exists():
        raise RuntimeError("Este necesar modelul final calibrat, nu pilotul.")
    FastDecisionModel.for_inference(model)
    predictions = runtime.predict_saved(data["test"], metadata["test"],
        lambda state, questions: FastDecisionModel.predict(model, tokenizer, state, questions),
        RUN_DIR / "test_predictions.jsonl", final_fingerprint + EXPECTED_MANIFEST_SHA)
    test_results = runtime.grouped_metrics(predictions)
    runtime.save_json(RUN_DIR / "test_metrics.json", test_results)
    print(json.dumps(test_results, ensure_ascii=False, indent=2))
else:
    print("Testul final nu este activat. Modelul salvat și checkpointurile nu sunt pierdute.")


## 10. Evaluare pe lotul suplimentar deja generat

Nu se generează alte articole și nu se mai calibrează modelul. Sunt aceleași 30 de surse noi,
două modele Qwen2.5 și două profiluri. Controalele umane sunt mai scurte; aceleași controale
sunt reutilizate în fiecare condiție. Amprenta folosită aici este a modelului tocmai antrenat,
nu cea a modelului istoric din primul experiment.


In [ ]:
from unsloth.models.clef import encode_record
supplement_data = [{'state':r['text'], 'questions':runtime.QUESTIONS, 'gold':{'origin':r['label']}} for r in supplement_rows]
supplement_metadata = [{k:v for k,v in r.items() if k!='text'} for r in supplement_rows]
text_tokenizer = getattr(tokenizer, 'tokenizer', tokenizer)
supplement_lengths = runtime.inspect_lengths({'supplement':supplement_data}, {'supplement':supplement_metadata},
    lambda row,budget: encode_record(text_tokenizer, row, max_length=budget), MAX_SEQ_LENGTH)
assert supplement_lengths['over_context'] == 0, 'Context insuficient; nu se trunchiază textele'
SUPPLEMENT_DIR = RUN_DIR / 'supplement'
runtime.save_json(SUPPLEMENT_DIR / 'token_lengths.json', supplement_lengths)
assert runtime.adapter_fingerprint(RUN_DIR / 'model_calibrated') == final_fingerprint
FastDecisionModel.for_inference(model)
supplement_predictions = runtime.predict_saved(supplement_data, supplement_metadata,
    lambda state,questions: FastDecisionModel.predict(model, tokenizer, state, questions),
    SUPPLEMENT_DIR / 'predictions.jsonl', final_fingerprint + hashlib.sha256(SUPPLEMENT_BUNDLE.read_bytes()).hexdigest())
assert len(supplement_predictions) == 150
supplement_results = {'overall':runtime.metrics(supplement_predictions), 'per_cell':{}, 'threshold':0.5,
    'model_fingerprint':final_fingerprint, 'caveats':supplement_protocol['source_limitations']}
human_controls = [r for r in supplement_predictions if r['label']=='human']
for generator in supplement_protocol['models']:
    for profile in supplement_protocol['profiles']:
        generated = [r for r in supplement_predictions if r['model']==generator and r['profile']==profile]
        assert len(generated) == 30
        supplement_results['per_cell'][generator+' / '+profile] = runtime.metrics(human_controls + generated)
runtime.save_json(SUPPLEMENT_DIR / 'metrics.json', supplement_results)
assert runtime.adapter_fingerprint(RUN_DIR / 'model_calibrated') == final_fingerprint
print(json.dumps(supplement_results, ensure_ascii=False, indent=2))


## 11. Rezultate și salvare

Indicatorii de mai jos provin din rularea curentă. Pentru fiecare generator se folosesc numai
sursele human pereche din test. Lotul suplimentar rămâne separat; nu însumăm subseturile
cu controale comune. Arhiva descărcată conține modelul calibrat și rezultatele, nu checkpointurile.


In [ ]:
from IPython.display import display, HTML
from html import escape
summary_rows = [('principal', test_results['overall'])] + list(test_results['per_model_matched_sources'].items())
summary_rows += [('supliment / '+name, values) for name,values in supplement_results['per_cell'].items()]
table = '<table><tr><th>Lot / generator</th><th>N</th><th>Acuratețe</th><th>FPR</th><th>Recall AI</th></tr>'
for name,values in summary_rows:
    table += '<tr><td>'+escape(name)+'</td><td>'+str(values['n'])+'</td><td>'+f"{values['accuracy']:.2%}"+'</td><td>'+f"{values['false_positive_rate']:.2%}"+'</td><td>'+f"{values['ai_recall']:.2%}"+'</td></tr>'
display(HTML(table+'</table>'))
runtime.save_json(RUN_DIR / 'reproducere.json', {'run_name':RUN_NAME, 'data_base':DATA_BASE,
    'main_data_sha256':hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest(),
    'supplement_data_sha256':hashlib.sha256(SUPPLEMENT_BUNDLE.read_bytes()).hexdigest(),
    'model_fingerprint':final_fingerprint, 'training_from_scratch':True, 'local_generation_required':False})
archive_path = RUN_DIR / 'experiment_colab.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIR.rglob('*')):
        relative = path.relative_to(RUN_DIR)
        if path.is_file() and path != archive_path and not any(part in {'checkpoints','model_uncalibrated'} for part in relative.parts):
            archive.write(path, relative.as_posix())
with zipfile.ZipFile(archive_path) as archive:
    assert archive.testzip() is None
print('Rezultate:', archive_path, '| MiB:', round(archive_path.stat().st_size/1024**2, 2))
from google.colab import files
files.download(str(archive_path))
